In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load training and test data
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

# Save original test for submission Ids
original_test = test.copy()

# Quick look
print(train.shape)
train.head()


In [ ]:
# Drop target from train
y_train = train['SalePrice']
X_train = train.drop(['SalePrice'], axis=1)

# Combine train and test for consistent encoding
combined = pd.concat([X_train, test], keys=['train', 'test'])

# One-hot encode categorical features
combined_encoded = pd.get_dummies(combined)

# Split back into train and test
X_train_encoded = combined_encoded.loc['train']
X_test_encoded = combined_encoded.loc['test']

# Align columns (ensure same features)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1)

# Impute missing values
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train_encoded)
X_test_imputed = imputer.transform(X_test_encoded)


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "XGBoost": xgb.XGBRegressor(),
    "LightGBM": lgb.LGBMRegressor(),
    "CatBoost": CatBoostRegressor(verbose=0)
}

# Evaluate models
for name, model in models.items():
    score = -cross_val_score(model, X_train_imputed, y_train, scoring="neg_root_mean_squared_error", cv=5).mean()
    print(f"{name}: RMSE = {score:.4f}")



In [ ]:
best_model = lgb.LGBMRegressor()
best_model.fit(X_train_imputed, y_train)

# Plot top 10 features
importances = pd.Series(best_model.feature_importances_, index=X_train_encoded.columns)
top10 = importances.sort_values(ascending=False).head(10)

top10.plot(kind='barh')
plt.title("Top 10 Feature Importances")
plt.show()


In [ ]:
# Predict on test set
preds = best_model.predict(X_test_imputed)

# Create submission file
submission = pd.DataFrame({
    "Id": original_test["Id"],
    "SalePrice": preds
})
submission.to_csv("submission.csv", index=False)


In [ ]:
# Select only numeric columns for correlation
numeric_train = train.select_dtypes(include=[np.number])

# Compute correlation matrix
corr = numeric_train.corr()

# Plot heatmap
plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap='coolwarm', annot=False)
plt.title("Feature Correlation Heatmap")
plt.show()
